In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"
os.environ['MKL_THREADING_LAYER'] = "GNU"

In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
from concept_abstraction.mimic import get_mimic_eval_info

import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
torch.cuda.set_per_process_memory_fraction(0.5)
torch.set_num_threads(1)
resource.setrlimit(resource.RLIMIT_AS, (30 * 1024 * 1024 * 1024, -1))


In [5]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [6]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 42
        environment_string = "mimic"
        gold_timesteps = 250_000
        training_timesteps = 250_000
        num_concepts_selected = 40
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = True
        run_iterative = False
        run_two_stage = False 
        run_imperfect=False
        run_intervention=False
        # Experiment #3
        cbm_accuracy_by_concept = None 
        intervention_probability = 0
        intervention_accuracy_by_concept = None 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 2
        selections_per_round = 1
        initial_concepts = 0
        out_folder = "llm"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--gold_timesteps', help='Number of training timesteps without concepts', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--intervention_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the two stage?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the iterative?', action='store_true')
        parser.add_argument('--run_intervention', help='Run the intervention?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--intervention_probability', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        gold_timesteps = args.gold_timesteps
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        run_intervention = args.run_intervention
        intervention_probability = args.intervention_probability
        intervention_accuracy_by_concept = args.intervention_accuracy_by_concept
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [7]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'gold_timesteps': gold_timesteps,
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
                'run_intervention': run_intervention,
                'run_imperfect': run_imperfect, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'mimic', 'training_timesteps': 250000, 'gold_timesteps': 250000, 'selection_function': 'q_value', 'num_concepts_selected': 40, 'cbm_accuracy_by_concept': None, 'cbm_std_by_concept': None, 'intervention_probability': 0, 'intervention_accuracy_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 2, 'selections_per_round': 1, 'initial_concepts': 0, 'run_basic': True, 'run_iterative': False, 'run_two_stage': False, 'run_intervention': False, 'run_imperfect': False}


In [8]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

In [9]:
import pickle
train_pkl = pickle.load(open("../../data/cub/train.pkl","rb"))
test_pkl = pickle.load(open("../../data/cub/test.pkl","rb"))
val_pkl = pickle.load(open("../../data/cub/val.pkl","rb"))

In [11]:
train_pkl[0]

{'id': 1210,
 'img_path': '/juice/scr/scr102/scr/thaonguyen/CUB_supervision/datasets/CUB_200_2011/images/022.Chuck_will_Widow/Chuck_Will_Widow_0059_796982.jpg',
 'class_label': 21,
 'attribute_label': [0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0],
 'attribute_certainty': [4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [10]:
first_chosen = [1,4,6,7]

In [12]:
dataset = json.load(open("../../data/cub/preprocessed.json"))

In [16]:
len(dataset['train']),len(dataset['test'])

(5994, 5794)

In [21]:
train_pkl[0]['img_path'].split("images/")[1]

'022.Chuck_will_Widow/Chuck_Will_Widow_0059_796982.jpg'

In [17]:
len(train_pkl), len(val_pkl),len(test_pkl)

(4796, 1198, 5794)

In [27]:
train_pkl[0]

{'id': 1210,
 'img_path': '/juice/scr/scr102/scr/thaonguyen/CUB_supervision/datasets/CUB_200_2011/images/022.Chuck_will_Widow/Chuck_Will_Widow_0059_796982.jpg',
 'class_label': 21,
 'attribute_label': [0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0],
 'attribute_certainty': [4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [30]:
[i for i in range(len(dataset['train'])) if 'Chuck_Will_Widow_0059_796982.jpg' in dataset['train'][i]['location']]

[649]

In [14]:
import numpy as np
from collections import defaultdict

# Read images.txt to map image_id to class_label and img_path
image_id_to_info = {}
with open('../../data/cub/images.txt', 'r') as f:
    for line in f:
        parts = line.strip().split()
        image_id = int(parts[0])
        img_path = parts[1]
        image_id_to_info[image_id] = img_path

# Create a mapping from img_path to image_id for easier lookup
img_path_to_id = {v: k for k, v in image_id_to_info.items()}

# Read attribute labels
# Structure: image_id -> attribute_id -> (is_present, certainty)
image_attributes = defaultdict(dict)
with open('../../data/cub/attributes/image_attribute_labels.txt', 'r') as f:
    for line in f:
        parts = line.strip().split()
        image_id = int(parts[0])
        attribute_id = int(parts[1]) - 1  # Convert to 0-indexed
        is_present = int(parts[2])
        certainty_id = int(parts[3])
        image_attributes[image_id][attribute_id] = (is_present, certainty_id)

# Collect attributes for each class based on train_pkl
# We'll count occurrences of 0 and 1 for each attribute, excluding certainty == 1
class_attr_count = np.zeros((200, 312, 2))

for entry in train_pkl:
    img_path = entry['img_path']
    class_label = entry['class_label']
    
    # Extract the relative path (last part after CUB_200_2011/images/)
    relative_path = img_path.split('CUB_200_2011/images/')[-1]
    
    # Get image_id from the relative path
    if relative_path in img_path_to_id:
        image_id = img_path_to_id[relative_path]
        
        # Get attributes for this image
        for attr_id in range(312):
            if attr_id in image_attributes[image_id]:
                is_present, certainty = image_attributes[image_id][attr_id]
                # Skip if attribute is not present (0) and certainty == 1 (not visible)
                if is_present == 0 and certainty == 1:
                    continue
                class_attr_count[class_label][attr_id][is_present] += 1

# Compute class attribute labels (argmax of counts)
class_attr_min_label = np.argmin(class_attr_count, axis=2)
class_attr_max_label = np.argmax(class_attr_count, axis=2)

# Where counts are equal (both 0), set label to 1
equal_count = np.where(class_attr_min_label == class_attr_max_label)
class_attr_max_label[equal_count] = 1

# Convert to float (this is your 200x312 array)
avg_attributes = class_attr_max_label.astype(float)

# Create subset array with selected attributes from 'ref'
# Round the values for the subset (though they're already binary)
subset_attributes = np.round(avg_attributes[:, ref]).astype(int)



In [23]:
def verify_dataset(pkl_data, dataset_name):
    mismatches = 0
    for i, entry in enumerate(pkl_data):
        class_label = entry['class_label']
        expected = subset_attributes[class_label]
        actual = np.array(entry['attribute_label'])
        
        if not np.array_equal(expected, actual):
            mismatches += 1
            if mismatches <= 3:  # Show first 3 mismatches
                print(f"{dataset_name} mismatch at index {i}:")
                print(f"  Expected: {expected[:10]}...")
                print(f"  Actual: {actual[:10]}...")
    
    if mismatches == 0:
        print(f"✓ {dataset_name}: All {len(pkl_data)} entries match!")
    else:
        print(f"✗ {dataset_name}: {mismatches}/{len(pkl_data)} mismatches")
    return mismatches == 0



In [24]:
train_match = verify_dataset(train_pkl, "train_pkl")
test_match = verify_dataset(test_pkl, "test_pkl")
val_match = verify_dataset(val_pkl, "val_pkl")


✓ train_pkl: All 4796 entries match!
✓ test_pkl: All 5794 entries match!
✓ val_pkl: All 1198 entries match!


In [29]:
def update_dataset(pkl_data):
    for entry in pkl_data:
        class_label = entry['class_label']
        # Update attribute_label with all 312 attributes
        entry['attribute_label'] = full_class_attributes[class_label].tolist()
        
        # Update path
        old_path = entry['img_path']
        relative_path = old_path.split('CUB_200_2011/images/')[-1]
        entry['img_path'] = f'/usr0/home/naveenr/projects/ConceptBottleneck/CUB_200_2011/images/{relative_path}'

update_dataset(train_pkl)
update_dataset(test_pkl)
update_dataset(val_pkl)


In [26]:
full_class_attributes = class_attr_max_label.astype(int)


In [32]:
pickle.dump(train_pkl, open("../../data/cub/train.pkl", "wb"))
pickle.dump(test_pkl, open("../../data/cub/test.pkl", "wb"))
pickle.dump(val_pkl, open("../../data/cub/val.pkl", "wb"))


In [43]:
np.mean(total_X[total_Y == 22][:,10])

0.7321428571428571

In [45]:
len(set([str(i['attribute_label']) for i in train_pkl]))

200

In [48]:
train_locations = set([i['img_path'].split("images/")[1] for i in train_pkl])

In [79]:
true_train = [i for i in dataset['train'] if i['location'] in train_locations]

In [80]:
train_X = np.array([row['attributes'] for row in true_train])
train_Y = np.array([row['label'] for row in true_train])


In [12]:
ref = [1, 4, 6, 7, 10, 14, 15, 20, 21, 23, 25, 29, 30, 35, 36, 38, 40, 44, 45, 50, 51, 53, 54, 56, 57, 59, 63, 64, 69, 70, 72, 75, 80, 84, 90, 91, \
    93, 99, 101, 106, 110, 111, 116, 117, 119, 125, 126, 131, 132, 134, 145, 149, 151, 152, 153, 157, 158, 163, 164, 168, 172, 178, 179, 181, \
    183, 187, 188, 193, 194, 196, 198, 202, 203, 208, 209, 211, 212, 213, 218, 220, 221, 225, 235, 236, 238, 239, 240, 242, 243, 244, 249, 253, \
    254, 259, 260, 262, 268, 274, 277, 283, 289, 292, 293, 294, 298, 299, 304, 305, 308, 309, 310, 311]

In [58]:
train_pkl[0]['attribute_label']

[0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0]

In [75]:
np.max(train_Y)

200

In [78]:
ref[3]

7

In [66]:
pred = np.mean(train_X[train_Y == 21],axis=0)[ref]
actual =train_pkl[0]['attribute_label']

In [69]:
data = train_pkl 
min_class_count = 10
N_CLASSES = 200
N_ATTRIBUTES=112
class_attr_count = np.zeros((N_CLASSES, N_ATTRIBUTES, 2))
for d in data:
    class_label = d['class_label']
    certainties = d['attribute_certainty']
    for attr_idx, a in enumerate(d['attribute_label']):
        if a == 0 and certainties[attr_idx] == 1: #not visible
            continue
        class_attr_count[class_label][attr_idx][a] += 1

class_attr_min_label = np.argmin(class_attr_count, axis=2)
class_attr_max_label = np.argmax(class_attr_count, axis=2)
equal_count = np.where(class_attr_min_label == class_attr_max_label) #check where 0 count = 1 count, set the corresponding class attribute label to be 1
class_attr_max_label[equal_count] = 1

attr_class_count = np.sum(class_attr_max_label, axis=0)
mask = np.where(attr_class_count >= min_class_count)[0] #select attributes that are present (on a class level) in at least [min_class_count] classes
class_attr_label_masked = class_attr_max_label[:, mask]
collapse_fn = lambda d: list(class_attr_label_masked[d['class_label'], :])

In [68]:
train_pkl[0]

{'id': 1210,
 'img_path': '/juice/scr/scr102/scr/thaonguyen/CUB_supervision/datasets/CUB_200_2011/images/022.Chuck_will_Widow/Chuck_Will_Widow_0059_796982.jpg',
 'class_label': 21,
 'attribute_label': [0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0],
 'attribute_certainty': [4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [67]:
[pred[i] for i in range(len(actual)) if actual[i]]

[0.8148148148148148,
 0.25925925925925924,
 0.2222222222222222,
 0.3333333333333333,
 0.14814814814814814,
 0.1111111111111111,
 0.25925925925925924,
 0.7037037037037037,
 0.9259259259259259,
 0.18518518518518517,
 0.1111111111111111,
 0.18518518518518517,
 0.2222222222222222,
 0.2222222222222222,
 0.8518518518518519,
 0.18518518518518517,
 0.0,
 0.18518518518518517,
 0.07407407407407407]

In [85]:
train_pkl[0]['attribute_label']

[0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0]

In [88]:
len()

112

In [117]:
[i['attribute_label'][3] for i in train_pkl if i['class_label'] == 21]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [95]:
d = dict(Counter([i['class_label'] for i in train_pkl]))

In [97]:
d[0]

26

In [127]:
[i['attribute_label'][3] for i in train_pkl if i['class_label'] == 21]


[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [131]:
true_train[0].keys()

dict_keys(['label', 'location', 'attributes'])

In [ ]:
class_attr_count = np.zeros((N_CLASSES, N_ATTRIBUTES, 2))
for d in data:
    class_label = d['class_label']
    certainties = d['attribute_certainty']
    for attr_idx, a in enumerate(d['attribute_label']):
        if a == 0 and certainties[attr_idx] == 1: #not visible
            continue
        class_attr_count[class_label][attr_idx][a] += 1


In [129]:
np.sum(train_X[train_Y == 22, 7])  # train_Y uses 1-200, so 22 maps to class_label 21


11.0

In [126]:
max([i['class_label'] for i in train_pkl])

199

In [124]:
np.max(train_Y)

200

In [122]:
np.mean(train_X[train_Y == 21,7])

0.8148148148148148

In [115]:
ref[3]

7

In [116]:
np.mean(train_X[train_Y == 22][:,7])

0.4782608695652174

In [110]:
np.mean(train_X[train_Y == 22],axis=0)[np.array(ref)][np.array(train_pkl[0]['attribute_label']) == 1]

array([0.47826087, 0.7826087 , 0.7826087 , 0.52173913, 0.65217391,
       0.60869565, 0.65217391, 0.47826087, 0.73913043, 0.69565217,
       0.56521739, 0.65217391, 0.56521739, 0.39130435, 0.56521739,
       0.86956522, 0.13043478, 0.73913043, 0.56521739])

In [46]:
dataset['train'][0]

{'label': 1,
 'location': '001.Black_footed_Albatross/Black_Footed_Albatross_0009_34.jpg',
 'attributes': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,

In [33]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])
train_Y = np.array([row['label'] for row in dataset['train']])
test_Y = np.array([row['label'] for row in dataset['test']])

# total_X = np.concatenate([train_X,test_X])
# total_Y = np.concatenate([train_Y,test_Y])

# rows_by_label = {}
# for label in set(total_Y):
#     relevant_subset = total_X[total_Y == label]
#     relevant_subset = np.round(np.mean(relevant_subset,axis=0))
#     rows_by_label[label] = relevant_subset

#     train_X[train_Y == label] = relevant_subset
#     test_X[test_Y == label] = relevant_subset


In [31]:
train_X[649,[1,4,6,7]]

array([0., 0., 0., 0.])

### Basic Setup

In [9]:
if is_main:
    concept_list = get_concepts(environment_string,concept_source,seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env, additional_info = get_environment(environment_string, None, seed)   

In [10]:
if is_main:
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    
    if os.path.exists(model_name):
        print("Model exists!")
        groundtruth_model = PPO.load(model_name)
        additional_info['subset_concepts'] = get_concepts(environment_string,"human_selected",seed)
    else:
        if "cyclic" in environment_string or "tree" in environment_string or "mimic" in environment_string:
            policy = "MlpPolicy"
        else:
            policy = "CnnPolicy"
        
        if environment_string == "mimic":
            additional_info['subset_concepts'] = get_concepts(environment_string,"human_selected",seed)
            groundtruth_model = train_ppo_model(ground_truth_env,"mimic_raw",total_timesteps=gold_timesteps,policy=policy,additional_info=additional_info)
        else:
            groundtruth_model = train_ppo_model(ground_truth_env,environment_string,total_timesteps=gold_timesteps,policy=policy)
        groundtruth_model.save(model_name)
    groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,groundtruth_model,seed)
    results['ground_truth'] = {'reward':groundtruth_reward}
    print("Basic:",results['ground_truth']['reward'])

Model exists!


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Basic: 1.8364820516874587


/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


### Basic Comparison

In [11]:
if is_main:    
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,training_timesteps,seed,selection_function,concept_source)
    results['basic_comparison'] = {}
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))
    else:
        if selection_function == "q_value":
            if environment_string == "mimic":
                modified_concepts = [lambda s, concept=concept: concept(additional_info['centers'][s]) 
                            for concept in concept_list]

                q_estimates = rollout_q_estimates_td(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),modified_concepts,learning_rate=1e-2,mimic=True,total_timesteps=10000,final_training=0)
            else:
                q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
        elif selection_function == "policy":
            if environment_string == "mimic":
                modified_concepts = [lambda s, concept=concept: concept(additional_info['centers'][s]) 
                            for concept in concept_list]

                q_estimates = rollout_pi_estimates(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),modified_concepts,mimic=True)
            else:
                q_estimates = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)
        pickle.dump(q_estimates,open(model_name,"wb"))

In [25]:
if is_main and run_basic:
    # Train a random policy
    if environment_string == "mimic":
        model = RandomAgent(GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])))
    else:
        model = RandomAgent(ground_truth_gym_env)
    random_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,model,seed)
    results['basic_comparison']['random'] = {'reward':random_reward}
    print("Random:",results['basic_comparison']['random']['reward'])

Random: 22.238180196253346


In [28]:
if is_main and run_basic:
    # Train a random selector
    subset_concept, random_idx = random_selection(concept_list,num_concepts_selected)
    subset_concept = [concept_list[i] for i in random_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)    
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",additional_info=additional_info)
    random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['random_selection'] = {'reward':random_selection_reward, 'concepts': random_idx}
    print("Random Selection:",results['basic_comparison']['random_selection']['reward'])

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


Step: 25024 | AvgR: 13.047 | OPE: 1.563 | ESS 0.16567318265808748 | Diff 0.607 | EV: 0.0 | VLoss: 81.35891914367676 | KL: 0.0016634240746498108 | ClipF: 0.0 | GradN: 0.49999994473575243
Step: 50048 | AvgR: -41.802 | OPE: 1.783 | ESS 0.13443664882958414 | Diff 0.570 | EV: -2.0599682331085205 | VLoss: 842.4138793945312 | KL: 8.665025234222412e-05 | ClipF: 0.0 | GradN: 0.5000000210789285
Step: 75072 | AvgR: 14.809 | OPE: 1.846 | ESS 0.14377111550440116 | Diff 0.627 | EV: -0.002595067024230957 | VLoss: 33.17166042327881 | KL: 3.3155083656311035e-07 | ClipF: 0.0 | GradN: 0.5000000366067177
Step: 100096 | AvgR: 10.388 | OPE: 1.918 | ESS 0.16143068685658502 | Diff 0.665 | EV: -0.19826102256774902 | VLoss: 48.468905448913574 | KL: 0.003847990185022354 | ClipF: 0.0 | GradN: 0.4999999974311434
Step: 125120 | AvgR: -19.874 | OPE: 1.625 | ESS 0.12390673295436479 | Diff 0.554 | EV: 0.0 | VLoss: 1.5871684849262238 | KL: 0.0 | ClipF: 0.0 | GradN: 0.49999997165581345
Step: 150144 | AvgR: 15.169 | OPE:

In [15]:
if is_main and run_basic:
    # Train a greedy selector
    subset_concept, greedy_idx = greedy_selection(concept_list,24,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    
    model = train_ppo_model(env,environment_string,total_timesteps=1_000_000,policy="MlpPolicy",additional_info=additional_info)
    greedy_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['greedy'] = {'concepts': greedy_idx, 'reward':greedy_selection_reward}
    print("Greedy:",results['basic_comparison']['greedy']['reward'])


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 106496 | AvgR: 306.660 | EV: -0.0030225515365600586 | VLoss: 132.93107795715332 | KL: 0.001301768934354186 | ClipF: 0.0 | GradN: 0.5000000137039535
Step: 212992 | AvgR: 383.330 | EV: -0.00018608570098876953 | VLoss: 93.03271999359131 | KL: 0.001434400212019682 | ClipF: 0.0 | GradN: 0.500000018015796
Step: 319488 | AvgR: 460.690 | EV: -0.0016169548034667969 | VLoss: 45.74244289398193 | KL: 0.0009686741977930069 | ClipF: 0.001171875 | GradN: 0.49999992866525905
Step: 425984 | AvgR: 440.360 | EV: 0.028561770915985107 | VLoss: 3.7849068254232408 | KL: 0.0030669220723211765 | ClipF: 0.0289306640625 | GradN: 0.49999994713032564
Step: 532480 | AvgR: 463.660 | EV: -0.01653432846069336 | VLoss: 28.412964391708375 | KL: 0.0019451524131000042 | ClipF: 0.0103271484375 | GradN: 0.49999947865460453
Step: 638976 | AvgR: 454.260 | EV: -0.011084318161010742 | VLoss: 18.553182458877565 | KL: 0.002228938741609454 | ClipF: 0.016064453125 | GradN: 0.4999998262723813
Step: 745472 | AvgR: 488.580 | EV:

In [13]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans


In [18]:
from math import ceil 

In [31]:
if is_main and run_basic and environment_string == "mimic":
    labels = KMeans(n_clusters=10).fit_predict(additional_info['centers'])
    tree = DecisionTreeClassifier(max_leaf_nodes=21)
    tree.fit(additional_info['centers'], labels)

    # Extract top 20 splits
    features = tree.tree_.feature
    thresholds = tree.tree_.threshold

    functions = []
    for idx in np.where(features >= 0)[0][:num_concepts_selected]:
        j, t = features[idx], thresholds[idx]
        quantile = np.mean(additional_info['centers'][:,j] < t)
        quantile *= 4
        quantile = round(quantile)
        quantile /=4 
        quantile = np.percentile(additional_info['centers'][:,j],quantile*100)
        functions.append(lambda x, j=j, t=quantile: int(x[j] > t))
    subset_concept = functions
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)    
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",additional_info=additional_info)
    decision_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    print(decision_selection_reward)
    # results['basic_comparison']['decision_tree_selection'] = {'reward':decision_selection_reward, 'concepts': []}
    # print("Decision Tree Selection:",results['basic_comparison']['decision_tree_selection']['reward'])

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


Step: 25024 | AvgR: -0.162 | OPE: 1.529 | ESS 0.15526017193473024 | Diff 0.570 | EV: 0.8206575065851212 | VLoss: 119.63927268981934 | KL: 1.1214986443519592e-05 | ClipF: 0.0 | GradN: 0.4999999067118191
Step: 50048 | AvgR: 9.646 | OPE: 1.593 | ESS 0.14913081078496032 | Diff 0.569 | EV: 0.7837806195020676 | VLoss: 80.96743774414062 | KL: 3.4518539905548096e-05 | ClipF: 0.0 | GradN: 0.5000000127343811
Step: 75072 | AvgR: 13.371 | OPE: 1.653 | ESS 0.14011194493333382 | Diff 0.558 | EV: -1.1920928955078125e-07 | VLoss: 101.30492973327637 | KL: 0.0 | ClipF: 0.0 | GradN: 0.4999999810204111
Step: 100096 | AvgR: 17.168 | OPE: 1.683 | ESS 0.17842648370874575 | Diff 0.634 | EV: -0.09926354885101318 | VLoss: 43.81343746185303 | KL: 7.105246186256409e-05 | ClipF: 0.0 | GradN: 0.49999998111595395
Step: 125120 | AvgR: 10.607 | OPE: 1.678 | ESS 0.16241206913432432 | Diff 0.664 | EV: 1.8358230590820312e-05 | VLoss: 5.301371857058257e-07 | KL: 0.0 | ClipF: 0.0 | GradN: 0.007103701011646834
Step: 150144 

In [32]:
if is_main and run_basic:
    subset_concept, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",additional_info=additional_info)
    lp_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['lp'] = {'concepts': lp_idx, 'reward': lp_selection_reward}
    print("LP Selection:",results['basic_comparison']['lp']['reward'])

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


Step: 25024 | AvgR: 21.635 | OPE: 1.573 | ESS 0.12436110884275718 | Diff 0.553 | EV: -1.1920928955078125e-07 | VLoss: 33.13383674621582 | KL: 5.898997187614441e-06 | ClipF: 0.0 | GradN: 0.5000000034287109
Step: 50048 | AvgR: 11.439 | OPE: 1.551 | ESS 0.14720141400086906 | Diff 0.594 | EV: 0.0 | VLoss: 83.41322135925293 | KL: 6.654858589172363e-05 | ClipF: 0.0 | GradN: 0.49999990960982627
Step: 75072 | AvgR: 12.367 | OPE: 1.677 | ESS 0.13441959538731962 | Diff 0.565 | EV: 0.0 | VLoss: 88.56962966918945 | KL: 3.594905138015747e-07 | ClipF: 0.0 | GradN: 0.4999999214026671
Step: 100096 | AvgR: 7.642 | OPE: 1.695 | ESS 0.13834321594747698 | Diff 0.572 | EV: 0.1595054268836975 | VLoss: 29.459760189056396 | KL: 0.0016800779849290848 | ClipF: 0.0 | GradN: 0.5000000001869105
Step: 125120 | AvgR: 9.418 | OPE: 1.812 | ESS 0.12876757062920804 | Diff 0.572 | EV: 0.0 | VLoss: 66.94500541687012 | KL: 7.692724466323853e-07 | ClipF: 0.0 | GradN: 0.49999993071200144
Step: 150144 | AvgR: 13.413 | OPE: 1.

In [42]:
seed += 1

In [45]:
additional_info['centers'][0,2]

1.7908847330364446

In [55]:
np.random.seed(42)
sample = np.random.random((5,5))
clusterer = MiniBatchKMeans(n_clusters=3,
                            max_iter=30,n_init=32).fit(sample)
centers = clusterer.cluster_centers_
centers

array([[0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864],
       [0.26475197, 0.15400445, 0.6319142 , 0.49594456, 0.52058506],
       [0.02058449, 0.96990985, 0.83244264, 0.21233911, 0.18182497]])

In [60]:
seed += 1

In [61]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans

# # Example: use unsupervised cluster labels as pseudo-target
labels = KMeans(n_clusters=10).fit_predict(additional_info['centers'])
tree = DecisionTreeClassifier(max_leaf_nodes=21)
tree.fit(additional_info['centers'], labels)

# Extract top 20 splits
features = tree.tree_.feature
thresholds = tree.tree_.threshold

functions = []
for idx in np.where(features >= 0)[0][:20]:
    j, t = features[idx], thresholds[idx]
    functions.append(lambda x, j=j, t=t: int(x[j] > t))
subset_concept = functions
env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)    
model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",additional_info=additional_info)
random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
results['basic_comparison']['decision_tree_selection'] = {'reward':random_selection_reward, 'concepts': random_idx}
print("Decision Tree Selection:",results['basic_comparison']['decision_tree_selection']['reward'])

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


Step: 25024 | AvgR: 10.319 | OPE: 1.550 | ESS 0.17720208981920535 | Diff 0.638 | EV: -0.03109443187713623 | VLoss: 37.345932960510254 | KL: 1.5195459127426147e-05 | ClipF: 0.0 | GradN: 0.4999999571546779
Step: 50048 | AvgR: 13.666 | OPE: 1.431 | ESS 0.2076368729036952 | Diff 0.667 | EV: -0.005304813385009766 | VLoss: 32.542673110961914 | KL: 7.743015885353088e-05 | ClipF: 0.0 | GradN: 0.4999999170536619
Step: 75072 | AvgR: 13.007 | OPE: 1.656 | ESS 0.1458498764506579 | Diff 0.577 | EV: 0.035821616649627686 | VLoss: 47.00364589691162 | KL: 0.00014525651931762695 | ClipF: 0.0 | GradN: 0.4999999508678845
Step: 100096 | AvgR: 12.508 | OPE: 1.291 | ESS 0.21597331613560833 | Diff 0.689 | EV: -0.05825686454772949 | VLoss: 31.256737232208252 | KL: 0.00011295266449451447 | ClipF: 0.0 | GradN: 0.49999993776730983
Step: 125120 | AvgR: 15.141 | OPE: 1.465 | ESS 0.23746306368898026 | Diff 0.682 | EV: 0.02175217866897583 | VLoss: 61.96904277801514 | KL: 6.370432674884796e-05 | ClipF: 0.0 | GradN: 0.

In [62]:
if is_main and run_basic and environment_string == "mimic":
    from sklearn.decomposition import PCA

    pca = PCA(n_components=num_concepts_selected)
    Z = pca.fit_transform(additional_info['centers'])  # X_train: (n_samples,47)
    medians = np.median(Z, axis=0)

    # create callable functions
    functions = []
    for i in range(num_concepts_selected):
        w = pca.components_[i]
        m = medians[i]
        functions.append(lambda x, w=w, m=m: int(np.dot(x, w) > m))

    # Train a PCA selector
    subset_concept = functions
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)    
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",additional_info=additional_info)
    random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    print(random_selection_reward)
    # results['basic_comparison']['pca_selection'] = {'reward':random_selection_reward, 'concepts': random_idx}
    # print("PCA Selection:",results['basic_comparison']['pca_selection']['reward'])

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


Step: 25024 | AvgR: -7.250 | OPE: 1.563 | ESS 0.12598748749610963 | Diff 0.543 | EV: 0.0 | VLoss: 19.11148452758789 | KL: 0.0 | ClipF: 0.0 | GradN: 0.5000000260862566
Step: 50048 | AvgR: 8.010 | OPE: 1.852 | ESS 0.1448355533605292 | Diff 0.779 | EV: -0.6626443862915039 | VLoss: 36.05514335632324 | KL: 0.0009908303618431091 | ClipF: 0.0 | GradN: 0.5000000747427131
Step: 75072 | AvgR: 129.933 | OPE: 1.316 | ESS 0.022804879270934064 | Diff 0.596 | EV: 0.3985567092895508 | VLoss: 1310.4415893554688 | KL: 3.0919909477233887e-07 | ClipF: 0.0 | GradN: 0.49999995243367934
Step: 100096 | AvgR: 12.476 | OPE: 1.822 | ESS 0.14879991278579371 | Diff 0.592 | EV: -0.13354623317718506 | VLoss: 56.39310169219971 | KL: 1.111254096031189e-05 | ClipF: 0.0 | GradN: 0.49999999717767146
Step: 125120 | AvgR: 13.268 | OPE: 1.688 | ESS 0.2141042458989445 | Diff 0.655 | EV: 0.05083352327346802 | VLoss: 31.874235153198242 | KL: 8.268654346466064e-05 | ClipF: 0.0 | GradN: 0.4999999258277913
Step: 150144 | AvgR: 13

In [31]:
if is_main and run_basic:
    subset_concept, multiple_idx = multiple_lp_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",additional_info=additional_info)
    multiple_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['multiple'] = {'concepts': multiple_idx, 'reward': multiple_selection_reward}
    print("Multiple Selection:",results['basic_comparison']['multiple']['reward'])

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:400: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


Step: 25024 | AvgR: 7.054 | OPE: 1.495 | ESS 0.178644793633927 | Diff 0.604 | EV: 0.06989467144012451 | VLoss: 34.12947940826416 | KL: 7.257983088493347e-05 | ClipF: 0.0 | GradN: 0.49999994648502377
Step: 50048 | AvgR: 13.098 | OPE: 1.598 | ESS 0.1494939254434691 | Diff 0.566 | EV: -0.08328711986541748 | VLoss: 26.547107219696045 | KL: 7.670372724533081e-05 | ClipF: 0.0 | GradN: 0.49999994962237365
Step: 75072 | AvgR: -4.219 | OPE: 1.577 | ESS 0.13454403578027557 | Diff 0.545 | EV: 0.3457808494567871 | VLoss: 50.20650100708008 | KL: 2.9802322387695312e-08 | ClipF: 0.0 | GradN: 0.49999992497796963
Step: 100096 | AvgR: 10.209 | OPE: 1.587 | ESS 0.19692027385952618 | Diff 0.616 | EV: 0.0 | VLoss: 119.78428840637207 | KL: 0.0002948381006717682 | ClipF: 0.0 | GradN: 0.49999995123719204
Step: 125120 | AvgR: 14.602 | OPE: 1.704 | ESS 0.1692705934043632 | Diff 0.601 | EV: 0.04684138298034668 | VLoss: 105.74919509887695 | KL: 1.3830140233039856e-05 | ClipF: 0.0 | GradN: 0.4999999709497178
Step:

### Imperfect Concept Predictors

In [17]:
if is_main and run_imperfect:
    results['inaccurate_comparison'] = {}


In [18]:
if is_main and run_imperfect:
    greedy_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for (func,acc) in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        subset_concept, greedy_idx = greedy_selection(modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        greedy_inaccurate_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_idx
        }

    if greedy_inaccurate_reward != {}:
        results['inaccurate_comparison']['greedy'] = greedy_inaccurate_reward
        print(greedy_inaccurate_reward)

In [19]:
if is_main and run_imperfect:
    lp_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        _, lp_idx = lp_based_selection(ground_truth_gym_env,modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        lp_inaccurate_reward[modification] = { 'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': lp_idx}

    if lp_inaccurate_reward != {}:
        results['inaccurate_comparison']['lp'] = lp_inaccurate_reward
        print(lp_inaccurate_reward)

In [47]:
if is_main and run_imperfect:
    multiple_lp_selection_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    subset_concept, multiple_idx = multiple_lp_selection(ground_truth_gym_env,modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    multiple_lp_selection_reward = {}
    multiple_lp_selection_reward['reward'] = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    multiple_lp_selection_reward['concepts'] = multiple_idx

    if multiple_lp_selection_reward != {}:
        results['inaccurate_comparison']['multiple_lp'] = multiple_lp_selection_reward
        print(multiple_lp_selection_reward)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 9.438 | EV: -0.12402045726776123 | VLoss: 6.645718261599541 | KL: 0.0005285162478685379 | ClipF: 0.0 | GradN: 0.4999999258017272
Step: 2048 | AvgR: 9.667 | EV: -0.0459367036819458 | VLoss: 6.7269322872161865 | KL: 0.01329878531396389 | ClipF: 0.054296875 | GradN: 0.49999966407662255
Step: 3072 | AvgR: 9.950 | EV: 0.0034577250480651855 | VLoss: 5.952751797437668 | KL: 0.0043501658365130424 | ClipF: 0.008203125 | GradN: 0.49999911189054175
Step: 4096 | AvgR: 10.240 | EV: -0.008330583572387695 | VLoss: 8.842490303516389 | KL: 0.008086392655968666 | ClipF: 0.028125 | GradN: 0.4999996649092149
Step: 5120 | AvgR: 10.640 | EV: -0.017305731773376465 | VLoss: 13.047454154491424 | KL: 0.0009268729481846094 | ClipF: 0.0 | GradN: 0.4999998894778255
Step: 6144 | AvgR: 10.840 | EV: -0.0034971237182617188 | VLoss: 10.296269637346267 | KL: 0.0013262471184134483 | ClipF: 0.0 | GradN: 0.49999973147788124
Step: 7168 | AvgR: 10.860 | EV: -0.047722697257995605 | VLoss: 11.266533076763153

KeyError: 'inaccurate_comparison'

In [21]:
if is_main and run_imperfect:
    imperfect_lp_selection_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        
        if modification == "continuous":
            subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,q_estimates,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='min')
        else:
            subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,q_estimates,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='max')
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        imperfect_lp_selection_reward[modification] = {}
        imperfect_lp_selection_reward[modification]['reward'] = evaluate_model(environment_string,eval_env,additional_info,model,seed)
        imperfect_lp_selection_reward[modification]['concepts'] = imperfect_idx

    if imperfect_lp_selection_reward != {}:
        results['inaccurate_comparison']['imperfect_lp'] = imperfect_lp_selection_reward
        print(imperfect_lp_selection_reward)

### Intervention

In [22]:
if is_main and run_intervention and intervention_accuracy_by_concept is not None:
    results['intervention_comparison'] = {}

In [23]:
if is_main and run_intervention:
    greedy_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,0,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
        else:
            continue 
        _, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
        subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)

        greedy_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_idx
        }

    if greedy_intervention_reward != {}:
        results['intervention_comparison']['greedy'] = greedy_intervention_reward
        print(greedy_intervention_reward)

In [24]:
if is_main and run_intervention:
    lp_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,0,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
        else:
            continue 
        _, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]

        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)

        lp_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': lp_idx
        }

    if lp_intervention_reward != {}:
        results['intervention_comparison']['lp'] = lp_intervention_reward
        print(lp_intervention_reward)

In [25]:
if is_main and run_intervention:
    multiple_lp_intervention_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,0,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
    subset_concept, multiple_idx = multiple_lp_selection(ground_truth_gym_env,modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)

    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    multiple_lp_intervention_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
    subset_concept = [modified_concept_predictors[i] for i in multiple_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    multiple_lp_intervention_reward['reward'] = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    multiple_lp_intervention_reward['concepts'] = multiple_idx

    if multiple_lp_intervention_reward != {}:
        results['intervention_comparison']['multiple_lp'] = multiple_lp_intervention_reward
        print(multiple_lp_intervention_reward)

In [26]:
if is_main and run_intervention:
    imperfect_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,0,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
        else:
            continue 
        subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,q_estimates,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='max')
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
        subset_concept = [modified_concept_predictors[i] for i in imperfect_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)

        imperfect_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': imperfect_idx
        }

    if imperfect_intervention_reward != {}:
        results['intervention_comparison']['imperfect_lp'] = imperfect_intervention_reward
        print(imperfect_intervention_reward)

### Iterative

In [52]:
if is_main and run_iterative:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    results['iterative'] = {}

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 11.000 | EV: -0.12179350852966309 | VLoss: 9.28954874277115 | KL: 0.014733283780515194 | ClipF: 0.078515625 | GradN: 0.49999988802885365
Step: 2048 | AvgR: 12.281 | EV: -0.09218168258666992 | VLoss: 14.653174543380738 | KL: 0.009612037800252438 | ClipF: 0.047265625 | GradN: 0.4999996263575632
Step: 3072 | AvgR: 14.860 | EV: 0.04367858171463013 | VLoss: 14.572086918354035 | KL: 0.006182464770972729 | ClipF: 0.040625 | GradN: 0.49999976604942076
Step: 4096 | AvgR: 16.570 | EV: 0.013621509075164795 | VLoss: 21.194579017162322 | KL: 0.0039267828688025475 | ClipF: 0.062109375 | GradN: 0.49999977353841635
Step: 5120 | AvgR: 17.810 | EV: -0.008924245834350586 | VLoss: 26.809803128242493 | KL: 0.0009323426056653261 | ClipF: 0.0109375 | GradN: 0.4999997332019191
Step: 6144 | AvgR: 18.430 | EV: 0.010829508304595947 | VLoss: 24.276914155483247 | KL: 0.0006842240691184998 | ClipF: 0.001953125 | GradN: 0.4999999121441756
Step: 7168 | AvgR: 18.680 | EV: -0.04369091987609863 | VLos

In [53]:
if is_main and run_iterative:
    if cbm_accuracy_by_concept is None:
        modified_concept_predictors = concept_list 
    else:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed+idx) for func,acc,idx in zip(concept_list,cbm_accuracy_by_concept,list(range(len(concept_list))))]
    rewards_iterative, concepts_iterative = iterative_selection(eval_env,gold_model,environment_string,modified_concept_predictors,num_iterations,selections_per_round,seed,training_timesteps=training_timesteps)
    results['iterative']['iterative_selection'] = {'reward': rewards_iterative, 'concepts': concepts_iterative}
    print(rewards_iterative)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 10.188 | EV: -0.056397318840026855 | VLoss: 7.734652933478356 | KL: 0.005481778644025326 | ClipF: 0.01640625 | GradN: 0.4999998631806544
Step: 2048 | AvgR: 10.240 | EV: -0.15528500080108643 | VLoss: 7.981095498800277 | KL: 0.0050144558772444725 | ClipF: 0.003125 | GradN: 0.49999972387304387
Step: 3072 | AvgR: 10.260 | EV: -0.002994537353515625 | VLoss: 7.9513607442379 | KL: 0.0019517862237989902 | ClipF: 0.0 | GradN: 0.4999993907665337
Step: 4096 | AvgR: 10.820 | EV: -0.0013213157653808594 | VLoss: 10.362503457069398 | KL: 0.0027250414714217186 | ClipF: 0.003125 | GradN: 0.4861891465115216
Step: 5120 | AvgR: 12.330 | EV: -0.004886627197265625 | VLoss: 16.237417471408843 | KL: 0.003927919548004866 | ClipF: 0.05234375 | GradN: 0.49999949154399703
Step: 6144 | AvgR: 13.760 | EV: 0.0026909708976745605 | VLoss: 19.498047876358033 | KL: 0.000962940277531743 | ClipF: 0.0 | GradN: 0.4999998779931992
Step: 7168 | AvgR: 14.680 | EV: 0.011650443077087402 | VLoss: 20.90200505256

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 10.500 | EV: -0.046933650970458984 | VLoss: 8.032275873422623 | KL: 0.004478626884520054 | ClipF: 0.0 | GradN: 0.49999987347337743
Step: 2048 | AvgR: 10.500 | EV: -0.12637364864349365 | VLoss: 8.917704963684082 | KL: 0.010342476889491081 | ClipF: 0.148828125 | GradN: 0.49999975675475167
Step: 3072 | AvgR: 10.800 | EV: 0.004778742790222168 | VLoss: 7.482996720075607 | KL: 0.0018694133032113314 | ClipF: 0.0 | GradN: 0.499999131433416
Step: 4096 | AvgR: 11.200 | EV: -0.00582277774810791 | VLoss: 8.529218900203706 | KL: 0.005579509772360325 | ClipF: 0.0 | GradN: 0.4999995967718625
Step: 5120 | AvgR: 11.910 | EV: 0.003013134002685547 | VLoss: 13.547104108333588 | KL: 0.005161941051483154 | ClipF: 0.043359375 | GradN: 0.49999943379037476
Step: 6144 | AvgR: 12.870 | EV: 0.020588397979736328 | VLoss: 16.048936426639557 | KL: 0.006866311654448509 | ClipF: 0.053125 | GradN: 0.49999968899322406
Step: 7168 | AvgR: 13.770 | EV: 0.1128043532371521 | VLoss: 14.533906030654908 | KL:

In [38]:
if is_main and run_iterative:
    if cbm_accuracy_by_concept is None:
        modified_concept_predictors = concept_list 
    else:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed+idx) for func,acc,idx in zip(concept_list,cbm_accuracy_by_concept,list(range(len(concept_list))))]

    num_concepts_selected = num_iterations*selections_per_round
    bayesian_reward, bayesian_idx = bayesian_iterative_selection(ground_truth_gym_env,environment_string,seed,modified_concept_predictors,num_iterations+1,num_concepts_selected,training_timesteps=training_timesteps)

    results['iterative']['bayesian'] = {
        'reward': bayesian_reward, 
        'concepts': bayesian_idx
    }
    print(bayesian_reward)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 9.729 | EV: 0.02197486162185669 | VLoss: 5.747423756122589 | KL: 0.003222605912014842 | ClipF: 0.0 | GradN: 0.49999984200203035
Step: 2048 | AvgR: 9.833 | EV: -0.04039144515991211 | VLoss: 7.783748182654381 | KL: 0.010661299340426922 | ClipF: 0.0796875 | GradN: 0.49999962024190947
Step: 3072 | AvgR: 9.990 | EV: -0.0030025243759155273 | VLoss: 6.344387337565422 | KL: 0.003963674418628216 | ClipF: 0.00859375 | GradN: 0.3763626776136752
Step: 4096 | AvgR: 10.030 | EV: -0.010143160820007324 | VLoss: 7.540875786542893 | KL: 0.0026981427799910307 | ClipF: 0.015625 | GradN: 0.49999924990530176
Step: 5120 | AvgR: 10.000 | EV: 0.005370914936065674 | VLoss: 8.118899124860764 | KL: 0.0064657945185899734 | ClipF: 0.043359375 | GradN: 0.49999918603592924
Step: 6144 | AvgR: 10.070 | EV: 0.0039403438568115234 | VLoss: 7.808240967988968 | KL: 0.0011475365608930588 | ClipF: 0.003125 | GradN: 0.49999937356154867
Step: 7168 | AvgR: 10.090 | EV: 0.015208780765533447 | VLoss: 7.746848714

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 10.479 | EV: 0.010110616683959961 | VLoss: 8.213785171508789 | KL: 0.002524554030969739 | ClipF: 0.0 | GradN: 0.4999999037590803
Step: 2048 | AvgR: 10.188 | EV: -0.003924369812011719 | VLoss: 7.926512986421585 | KL: 0.0029223174788057804 | ClipF: 0.0 | GradN: 0.4999996931464959
Step: 3072 | AvgR: 10.030 | EV: -0.038591980934143066 | VLoss: 6.735201787948609 | KL: 0.003943895921111107 | ClipF: 0.003125 | GradN: 0.49999967738886686
Step: 4096 | AvgR: 10.030 | EV: -0.005759716033935547 | VLoss: 6.80656995177269 | KL: 0.0035683459136635065 | ClipF: 0.0 | GradN: 0.49999920683908655
Step: 5120 | AvgR: 9.960 | EV: 0.01065915822982788 | VLoss: 8.826655226945878 | KL: 0.003479492384940386 | ClipF: 0.0296875 | GradN: 0.4999995583960278
Step: 6144 | AvgR: 10.210 | EV: 0.00531005859375 | VLoss: 7.7076284945011135 | KL: 0.005521765910089016 | ClipF: 0.01015625 | GradN: 0.49999964995919755
Step: 7168 | AvgR: 10.130 | EV: 0.016875922679901123 | VLoss: 7.744528406858445 | KL: 0.0039

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

Step: 1024 | AvgR: 9.938 | EV: -0.0853116512298584 | VLoss: 7.336046609282493 | KL: 0.00045614270493388176 | ClipF: 0.0 | GradN: 0.49999989247503573
Step: 2048 | AvgR: 10.083 | EV: -0.06749379634857178 | VLoss: 7.6559153437614444 | KL: 0.006215488538146019 | ClipF: 0.041015625 | GradN: 0.4999996733405825
Step: 3072 | AvgR: 10.050 | EV: 0.010752081871032715 | VLoss: 6.091243094205856 | KL: 0.0014123825822025537 | ClipF: 0.0 | GradN: 0.49999944327144097
Step: 4096 | AvgR: 10.000 | EV: 0.014495611190795898 | VLoss: 6.9582974195480345 | KL: 0.012411911971867085 | ClipF: 0.028125 | GradN: 0.49999968128281036
Step: 5120 | AvgR: 10.030 | EV: -0.006209850311279297 | VLoss: 9.26285199522972 | KL: 0.0019203606061637402 | ClipF: 0.001171875 | GradN: 0.4999990343768039
Step: 6144 | AvgR: 9.960 | EV: -0.01311039924621582 | VLoss: 7.4328044831752775 | KL: 0.0014054377097636461 | ClipF: 0.0 | GradN: 0.499999744367808
Step: 7168 | AvgR: 9.990 | EV: 0.0070664286613464355 | VLoss: 7.028459107875824 | KL

KeyError: 'iterative'

### Two-Stage Training

In [30]:
if is_main and run_two_stage:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

In [31]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,list(range(len(concept_list))))
    results['two_stage'] = {}
    results['two_stage']['accuracy'] = acc_list 


In [32]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_two_stage = {}
    greedy_concepts, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,greedy_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    greedy_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy'] = {'reward': greedy_two_stage_reward, 'concepts': greedy_idx}
    print(greedy_two_stage_reward)



In [33]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_iterative_concepts, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,greedy_iterative_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    greedy_iterative_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy_iterative'] = {'reward': greedy_iterative_two_stage_reward, 'concepts': greedy_iterative_idx}
    print(greedy_iterative_two_stage_reward)



In [34]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    lp_concepts, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,lp_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, lp_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    lp_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['lp'] = {'reward': lp_two_stage_reward, 'concepts': lp_idx}
    print(lp_two_stage_reward)



In [35]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    top_k_idx = np.argsort(acc_list)[-num_concepts_selected:]
    top_k_concepts = [concept_list[i] for i in top_k_idx]
    
    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,top_k_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, top_k_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    top_k_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['top_k'] = {'reward': top_k_two_stage_reward, 'concepts': top_k_idx}
    print(top_k_two_stage_reward)



In [36]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    imperfect_concepts, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,concept_list,q_estimates,selection_function,target_abstraction,num_concepts_selected,acc_list,concept_source,environment_string,additional_info,direction='max')
    
    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,imperfect_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, imperfect_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    imperfect_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['imperfect'] = {'reward': imperfect_two_stage_reward, 'concepts': imperfect_idx}
    print(imperfect_two_stage_reward)

## Ablations

### Reward Perturbation

In [37]:
if is_main and reward_error > 0:
    results['reward_error'] = {}
    perturbed_groundtruth_eval_env = RewardPerturbationWrapper(ground_truth_gym_env,reward_error)

    if selection_function == "q_value":
        if environment_string == "mimic":
            q_estimates_perturbed = rollout_q_estimates_td(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),concept_list,learning_rate=1e-3,mimic=True,total_timesteps=5000,final_training=0)
        else:
            q_estimates_perturbed = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    elif selection_function == "policy":
        if environment_string == "mimic":
            q_estimates_perturbed = rollout_pi_estimates(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),concept_list,mimic=True)
        else:
            q_estimates_perturbed = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)

    subset_concept, greedy_perturbed_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    perturbed_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,perturbed_model,seed)
    results['reward_error']['greedy'] = {
        'reward': greedy_selection_perturbed_reward,
        'concepts': greedy_perturbed_idx
    }
    print(greedy_selection_perturbed_reward)

In [38]:
if is_main and reward_error > 0:
    subset_concept, greedy_perturbed_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_iterative_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['reward_error']['greedy_iterative'] = {
        'reward': greedy_iterative_selection_perturbed_reward,
        'concepts': greedy_perturbed_iterative_idx
    }
    print(greedy_iterative_selection_perturbed_reward)

In [39]:
if is_main and reward_error > 0:
    subset_concept, lp_perturbed_idx = lp_based_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    lp_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['reward_error']['lp'] = {
        'reward': lp_selection_perturbed_reward,
        'concepts': lp_perturbed_idx
    }
    print(lp_selection_perturbed_reward)

### Comparison with Concept Completeness

In [40]:
# TODO: Create a Shapley-based baseline
if is_main and assess_completeness:
    pass 


## Save Data

In [ ]:
if is_main:
    save_path = get_save_path(out_folder,save_name)

In [ ]:
if is_main:
    delete_duplicate_results(out_folder,"",results)

In [ ]:
if is_main:
    json.dump(results,open('../../results/'+save_path,'w'))